# Agent AML/PPE Research — ScreenEdge Africa
**Basé sur le TP agentAI.ipynb**

Stack :
- LLM : Ollama (llama3.1:8b) — local, gratuit
- Mémoire : InMemorySaver + thread_id
- Middlewares : wrap_model_call, wrap_tool_call, dynamic_prompt
- Tools : Tavily (web) + PyPDFLoader (PDFs) + WebBaseLoader (pages web) + GAFI checker
- Output : PPEReport (Pydantic)

## Cellule 1 — Imports

In [1]:
# Imports — version TP avec langchain 1.3.1
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, wrap_tool_call, dynamic_prompt, ModelRequest, ModelResponse
from langchain.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_ollama import ChatOllama
from langchain_tavily import TavilySearch
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader
from pydantic import BaseModel
from dotenv import load_dotenv
from IPython.display import Markdown, display
import os

print("✅ Imports OK")

C:\Users\pc\AppData\Local\Temp\ipykernel_30808\881623753.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


✅ Imports OK


## Cellule 2 — Variables d'environnement

In [ ]:
load_dotenv(override=True)

assert os.getenv('TAVILY_API_KEY'), '❌ TAVILY_API_KEY manquante dans .env'
print('✅ Variables d\'environnement chargées')

## Cellule 3 — LLM (Ollama llama3.1:8b)

In [ ]:
llm = ChatOllama(
    model='llama3.1:8b',
    temperature=0.1
)

print('✅ LLM Ollama llama3.1:8b chargé')

## Cellule 4 — System Prompt AML/PPE (Skill v8.1)

In [ ]:
AML_SYSTEM_PROMPT = """
Tu es un agent expert en AML/CFT et PPE (Personnes Politiquement Exposees)
specialise sur le Maghreb et l'Afrique de l'Ouest.

## ETAPE 1 - OBLIGATOIRE AVANT TOUTE REPONSE
Pour CHAQUE question concernant un pays, tu DOIS appeler EN PREMIER
le tool consulter_referentiel(pays=...) avant toute autre action.
Ne reponds JAMAIS sans avoir consulte le referentiel. Sans exception.

## WORKFLOW OBLIGATOIRE
Pour chaque requete PPE/AML, tu suis ces 4 phases :
1. [OBLIGATOIRE] Appeler consulter_referentiel(pays=...) -> donnees officielles
2. Extraire les elements critiques PPE (fonctions, famille, duree, obligations)
3. Verifier la fiabilite de la source (OFFICIEL > SECONDAIRE > NON FIABLE)
4. Produire un rapport structure JSON

## COUVERTURE GEOGRAPHIQUE
- Maghreb : Maroc (MA), Algerie (DZ), Tunisie (TN), Libye (LY)
- UEMOA : Senegal (SN), Cote d'Ivoire (CI), Togo (TG), Benin (BJ),
          Mali (ML), Burkina Faso (BF), Niger (NE), Guinee-Bissau (GW)
- Autres : Guinee (GN)

## PRINCIPE UEMOA UNIFIE
Les 8 pays UEMOA partagent la meme definition PPE via Directive n001-2023-CM.
Definition : 3 categories (etrangeres / nationales / orgs internationales)
9 fonctions (i a ix) + famille (x) + proches associes (xi) + RBA (xii)
Reevaluation obligatoire : tous les 3 ans

## STATUTS GAFI (fev. 2026)
- Liste grise : Algerie (DZ), Cote d'Ivoire (CI) -> VIGILANCE RENFORCEE OBLIGATOIRE
- Liste noire : RPDC, Iran, Myanmar UNIQUEMENT
- GAFI publie 3x/an apres chaque pleniere : fev/mars, juin/juil, oct/nov

## SOURCES OFFICIELLES PRIORITAIRES
- Maroc : bkam.ma / utrf.ma / ammc.ma
- Algerie : bank-of-algeria.dz (Instructions CTRF n02+03/2023)
- Tunisie : bct.gov.tn / ctaf.gov.tn
- UEMOA : centif.sn / centif.ci / centif.tg / centif.bj / centif.bf / centif.ne
- GAFI : fatf-gafi.org

## REGLES CRITIQUES
1. Appeler consulter_referentiel en PREMIER - toujours
2. Toujours citer la source officielle issue du referentiel
3. Gap documentaire -> appliquer fallback GAFI R12
4. Pays liste grise -> vigilance renforcee OBLIGATOIRE
"""

print('OK System prompt AML/PPE v8.1 charge (consulter_referentiel obligatoire)')


## Cellule 5 — Output Structuré (Pydantic)

In [ ]:
class PPEReport(BaseModel):
    pays: str
    code_iso: str
    zone: str                    # maghreb | uemoa | autre
    loi_reference: str
    statut_gafi: str             # clean | liste_grise | liste_noire
    vigilance_niveau: str        # standard | renforcee | maximale
    pep_fonctions: list
    famille_incluse: bool
    gap_detecte: bool
    action_recommandee: str
    source_url: str
    date_extraction: str
    prochaine_verification: str

print('✅ PPEReport Pydantic défini')

## Cellule 6 — Tools AML/PPE

In [ ]:
import os

SKILL_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "aml-pep-research-agent (1).md")

tavily_search_tool = TavilySearch(max_results=5, search_depth='advanced')

@tool
def consulter_referentiel(pays: str) -> str:
    """Consulte le referentiel officiel AML/PPE ScreenEdge pour un pays donne.
    Doit etre appele EN PREMIER avant toute reponse sur un pays."""
    try:
        with open(SKILL_PATH, 'r', encoding='utf-8') as f:
            contenu = f.read()
        pays_upper = pays.upper()
        marker = f"## {pays_upper}"
        if marker not in contenu:
            for line in contenu.split('\n'):
                if line.startswith('## ') and pays.lower() in line.lower():
                    marker = line.strip()
                    break
            else:
                return f'Pays "{pays}" non trouve dans le referentiel. Appliquer fallback GAFI R12.'
        debut = contenu.index(marker)
        fin = contenu.find('\n---', debut + len(marker))
        section = contenu[debut:fin if fin != -1 else debut + 3000]
        print(f'Referentiel consulte : {pays} ({len(section)} caracteres)')

        # Detection automatique du niveau de risque dans la section extraite
        section_lower = section.lower()
        needs_gafi_check = "liste_grise" in section_lower or "liste grise" in section_lower
        needs_enhanced = "renforcee" in section_lower or "renforcee obligatoire" in section_lower

        if needs_gafi_check:
            print(f'Pays liste grise detecte -> verification GAFI obligatoire')
            section += "\n\n[INSTRUCTION AUTOMATIQUE] Ce pays est en LISTE GRISE GAFI. Appelle verifier_statut_gafi() immediatement pour confirmer le statut actuel. Vigilance renforcee obligatoire."
        elif needs_enhanced:
            print(f'Vigilance renforcee detectee -> verification GAFI recommandee')
            section += "\n\n[INSTRUCTION AUTOMATIQUE] Ce pays necessite une vigilance renforcee. Appelle verifier_statut_gafi() pour verifier le statut GAFI actuel."

        return section
    except FileNotFoundError:
        return f'Fichier referentiel introuvable : {SKILL_PATH}'

@tool
def search_web(query: str) -> str:
    """Recherche des informations AML/PPE sur le web."""
    return tavily_search_tool.invoke({'query': query})

@tool
def lire_pdf_officiel(url: str) -> str:
    """Lit un PDF officiel depuis une URL (CENTIF, BCEAO, BAM, etc.)."""
    try:
        loader = PyPDFLoader(url)
        pages = loader.load()
        contenu = '\n'.join([p.page_content for p in pages[:10]])
        return contenu[:5000]
    except Exception as e:
        return f'Erreur lecture PDF : {str(e)}'

@tool
def lire_page_web(url: str) -> str:
    """Lit le contenu d'une page web officielle (CENTIF, autorites AML)."""
    try:
        loader = WebBaseLoader(url)
        docs = loader.load()
        return docs[0].page_content[:5000]
    except Exception as e:
        return f'Erreur lecture page web : {str(e)}'

@tool
def verifier_statut_gafi(pays: str) -> str:
    """Verifie le statut GAFI actuel d'un pays sur fatf-gafi.org."""
    query = f'FATF GAFI {pays} grey list black list 2026 site:fatf-gafi.org'
    return tavily_search_tool.invoke({'query': query})

# consulter_referentiel en premier - c'est le point d'entree obligatoire
tools_aml = [consulter_referentiel, search_web, lire_pdf_officiel, lire_page_web, verifier_statut_gafi]

print(f'OK {len(tools_aml)} tools AML/PPE definis')
for t in tools_aml:
    print(f'   - {t.name}')


## Cellule 7 — Middlewares (wrap_tool_call + dynamic_prompt)

In [ ]:
# Middleware 1 - Gestion erreurs tools (meme que le TP)
@wrap_tool_call
def handle_tool_errors(request, handler):
    """Gere les erreurs des tools avec message propre."""
    try:
        return handler(request)
    except Exception as e:
        print(f"Tool error: {e}")
        return ToolMessage(
            content=f"Erreur tool : {str(e)}. Essaie une autre approche.",
            tool_call_id=request.tool_call["id"]
        )

# Middleware 2 - Routing dynamique AML/PPE
@dynamic_prompt
def routing_aml(request: ModelRequest) -> str:
    """Force consulter_referentiel en premier, puis routing selon le type de requete."""
    messages = request.messages
    last_message = messages[-1].content.lower() if messages else ""

    # Instruction referentiel - toujours presente, quelle que soit la requete
    instruction_referentiel = """
RAPPEL OBLIGATOIRE : Appelle consulter_referentiel(pays=...) EN PREMIER.
Ne fournis aucune information sur un pays sans avoir consulte le referentiel avant.
"""

    # Triggers verification GAFI
    triggers_gafi = ["statut gafi", "liste grise", "liste noire",
                     "vigilance", "risque pays", "sanctionne", "fatf"]

    # Triggers lecture PDF
    triggers_pdf = ["pdf", "circulaire", "ordonnance", "loi ", "texte"]

    if any(t in last_message for t in triggers_gafi):
        print("Routing: GAFI check active")
        return AML_SYSTEM_PROMPT + instruction_referentiel + """
INSTRUCTION SPECIALE : Cette requete necessite une verification GAFI.
Etape 1 -> consulter_referentiel | Etape 2 -> verifier_statut_gafi"""

    elif any(t in last_message for t in triggers_pdf):
        print("Routing: Lecture PDF activee")
        return AML_SYSTEM_PROMPT + instruction_referentiel + """
INSTRUCTION SPECIALE : Cette requete necessite la lecture d'un document officiel.
Etape 1 -> consulter_referentiel | Etape 2 -> lire_pdf_officiel ou lire_page_web"""

    else:
        print("Routing: Recherche generale AML")
        return AML_SYSTEM_PROMPT + instruction_referentiel

print("OK Middlewares definis")


## Cellule 8 — Création de l'agent

In [ ]:
# Mémoire — même que le TP
memory = InMemorySaver()

# Création agent — même structure que le TP
agent_aml = create_agent(
    model=llm,
    tools=tools_aml,
    middleware=[handle_tool_errors, routing_aml],
    system_prompt=AML_SYSTEM_PROMPT,
    checkpointer=memory
)

print("✅ Agent AML/PPE créé !")
print(f"   LLM    : llama3.1:8b (Ollama local)")
print(f"   Tools  : {[t.name for t in tools_aml]}")
print(f"   Mémoire: InMemorySaver")

## Cellule 9 — Test 1 : Définition PPE Sénégal

In [ ]:
config = {'configurable': {'thread_id': 'session_test_1'}}

resp = agent_aml.invoke(
    {'messages': [HumanMessage('Quelle est la définition des PPE au Sénégal ?')]},
    config=config
)

display(Markdown(resp['messages'][-1].content))

## Cellule 10 — Test 2 : Mémoire (Sénégal → Côte d'Ivoire)

In [ ]:
# Même thread_id → l'agent se souvient de la session précédente
resp2 = agent_aml.invoke(
    {'messages': [HumanMessage("Et pour la Côte d'Ivoire, c'est pareil ?")]},
    config=config
)

display(Markdown(resp2['messages'][-1].content))

## Cellule 11 — Test 3 : Vérification statut GAFI (routing activé)

In [ ]:
# Le mot 'statut GAFI' déclenche le routing → verifier_statut_gafi activé
config_gafi = {'configurable': {'thread_id': 'session_gafi_1'}}

resp3 = agent_aml.invoke(
    {'messages': [HumanMessage("Quel est le statut GAFI de l'Algérie en 2026 ?")]},
    config=config_gafi
)

display(Markdown(resp3['messages'][-1].content))

## Cellule 12 — Test 4 : Lecture PDF officiel CENTIF

In [ ]:
# Le mot 'ordonnance' déclenche le routing → lire_pdf_officiel activé
config_pdf = {'configurable': {'thread_id': 'session_pdf_1'}}

resp4 = agent_aml.invoke(
    {'messages': [HumanMessage(
        'Lis le PDF de l\'ordonnance ivoirienne et extrait la définition PPE : '
        'https://www.centif.ci/wp-content/uploads/2025/03/ORDONNANCE-N2023-875-DU-23-NOVEMBRE-2023-RELATIVE-A-LA-LUTTE-CONTRE-LES-BLANCHIMENTS-DE-CAPIT.pdf'
    )]},
    config=config_pdf
)

display(Markdown(resp4['messages'][-1].content))

## Cellule 13 — Sessions multiples (clients isolés)

In [ ]:
# Chaque client a son thread_id → sessions complètement isolées
config_client_maroc   = {'configurable': {'thread_id': 'client_MA_001'}}
config_client_senegal = {'configurable': {'thread_id': 'client_SN_002'}}

resp_ma = agent_aml.invoke(
    {'messages': [HumanMessage('Je travaille sur un dossier PPE au Maroc')]},
    config=config_client_maroc
)

resp_sn = agent_aml.invoke(
    {'messages': [HumanMessage('Je travaille sur un dossier PPE au Sénégal')]},
    config=config_client_senegal
)

print('=== CLIENT MAROC ===')
display(Markdown(resp_ma['messages'][-1].content))

print('\n=== CLIENT SÉNÉGAL ===')
display(Markdown(resp_sn['messages'][-1].content))